# Instagram export — make an anonymised copy

This notebook turns the raw export into a copy that can leave your computer: the identifier files are
dropped, every person becomes a stable pseudonym (`user_0001`, …, and you become `me`), message text is
reduced to metadata unless you decide otherwise, location and device tags are stripped, links are reduced
to their domain. The output is written both as a JSON tree with the original layout and as tidy CSV tables.

As in notebook 01, everything runs inside your browser; nothing is uploaded. Before you start, the
`instagram-….zip` you downloaded from Instagram must be in the file browser on the left (drag it in if it isn't).

Run the cells top to bottom with **Shift+Enter**.

In [ ]:
# 1. Find the export (same as in notebook 01).
import os, sys, glob

if not os.path.exists("ig_anonymise.py"):
    for root, _dirs, files in os.walk("/drive"):
        if "ig_anonymise.py" in files:
            os.chdir(root)
            break
sys.path.insert(0, os.getcwd())

exports = sorted(glob.glob("*.zip")) + sorted(d for d in os.listdir() if d.startswith("instagram-") and os.path.isdir(d))
exports = [e for e in exports if not e.startswith(("report", "anonymised"))]
if exports:
    EXPORT = exports[0]
    print("Export found:", EXPORT)
    if len(exports) > 1:
        print("Several exports found; using the first one. Change EXPORT = ... to pick another:", exports)
else:
    EXPORT = None
    print("Nothing found yet: drag your instagram-....zip into the file browser on the left, then re-run this cell.")

## 2. Choose what happens to message text

Private messages are the sensitive part: the people you wrote with never agreed to be in a dataset, and
automatic redaction cannot make their words safe. Three policies:

| `TEXT` | what is kept |
|---|---|
| `"metadata"` **(default — use this for anything you share)** | every message keeps sender, time, length in characters and words, emoji count, link / mention / question flags, attachment type and reactions. No text at all. |
| `"own"` | your own messages are kept (with e-mails, phones, addresses… replaced by placeholders); other people's are reduced to metadata. |
| `"all"` | all text kept, redacted and pseudonymised. Names of people outside your network are **not** caught — read everything before sharing. |

Captions, your comments and hashtags are public already, so they are kept in every policy (pseudonymised and
redacted). `SHIFT_DAYS = True` moves every timestamp by the same hidden number of days: dates are hidden,
intervals, weekdays and hours are preserved.

In [ ]:
TEXT = "metadata"      # "metadata" | "own" | "all"
SHIFT_DAYS = False     # True -> shift all timestamps by a random 30-400 days (same shift everywhere)

In [ ]:
# 3. Anonymise. The last part of the report re-scans the output with the scanner from notebook 01:
#    it should say "HIGH findings: 0" and "non-pseudonym names still present: 0".
from ig_anonymise import anonymise, pack

result = anonymise(EXPORT, "anonymised", text=TEXT, shift_days=SHIFT_DAYS)
print(result.report())

**Reading the report.** *Dropped files* are gone entirely (login history, contacts, devices, inferred
locations, ad data, photos…). *Kept files* were rewritten with pseudonyms. *Transformations* counts what was
changed inside them. The *re-scan* is the proof: the scanner that found your e-mail, phone and IP addresses in
notebook 01 finds nothing of the kind in the output, and every name it sees is a pseudonym.

If the re-scan lists HIGH findings or non-pseudonym names, look at them in `anonymised/anonymisation_report.md`
before sharing anything (with `TEXT = "all"` this is expected for names of people outside your network).

In [ ]:
# 4. Pack the shareable part (json/, csv/, report) into one zip to download.
#    pseudonym_map.csv is NOT included: it links pseudonyms back to real people and stays with you.
z = pack("anonymised")
print("Packed:", z)
print("Right-click `anonymised.zip` in the file browser on the left -> Download.")
print("Keep `anonymised/pseudonym_map.csv` private (download it separately if you want to be able to decode).")

In [ ]:
# 5. Optional: a first look at the tables (pandas downloads into the browser on first import).
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("pandas could not be loaded (no internet connection?). The CSVs are in anonymised/csv/.")
else:
    msgs = pd.read_csv("anonymised/csv/messages.csv", parse_dates=["timestamp"])
    print(f"{len(msgs):,} messages in {msgs['thread'].nunique()} threads; "
          f"{(msgs['is_me']).mean():.0%} sent by you")
    display(msgs.groupby("thread").size().sort_values(ascending=False).head(10).rename("messages"))
    display(msgs.assign(hour=msgs["timestamp"].dt.hour).groupby("hour").size().rename("messages by hour of day (UTC)"))

## What you have now

`anonymised.zip` contains:

- `csv/` — `messages.csv`, `threads.csv`, `followers.csv`, `following.csv`, `posts.csv`, `comments.csv`,
  `likes.csv`, `saved.csv`, `stories.csv`… one row per event, pseudonyms only. This is the dataset for the
  analysis notebooks (it can go to Colab, a shared drive, or a classmate).
- `json/` — the anonymised export in Instagram's own layout, for code that expects the raw structure.
- `anonymisation_report.md` — what was dropped and changed, with the re-scan result.

**Clean up when you are done:** right-click the raw zip, `report/`, `anonymised/` and the zips → *Delete*,
or clear this site's data in your browser settings.